In [ ]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd
import time

load_dotenv()
KEY = os.getenv('OMDB_KEY')

In [9]:
def buscar_episodio(temporada, episodio, chave):
    url = 'https://www.omdbapi.com/'
    params = {
        't':       'Friends',
        'Season':  temporada,
        'Episode': episodio,
        'apikey':  chave
    }
    resposta = requests.get(url, params=params)
    dados = resposta.json()

    # Retorna None se o episódio não existir
    if dados.get('Response') == 'False':
        return None

    return {
        'temporada':   int(temporada),
        'episodio':    int(episodio),
        'titulo':      dados.get('Title'),
        'data_estreia': dados.get('Released'),
        'rating_imdb': dados.get('imdbRating'),
        'votos_imdb':  dados.get('imdbVotes'),
        'duracao':     dados.get('Runtime'),
        'plot':        dados.get('Plot')
    }

In [ ]:
todos_episodios = []

# Friends tem 10 temporadas
for temporada in range(1, 11):
    print(f'Baixando temporada {temporada}...')
    
    # Cada temporada tem entre 18 e 25 episódios
    for episodio in range(1, 26):
        resultado = buscar_episodio(temporada, episodio, KEY)
        
        if resultado is None:
            break  # chegou no fim da temporada, passa pra próxima
        
        todos_episodios.append(resultado)
        time.sleep(0.2)  # pausa pequena para não sobrecarregar a API

print(f'\nTotal de episódios coletados: {len(todos_episodios)}')

Baixando temporada 1...
Baixando temporada 2...
Baixando temporada 3...
Baixando temporada 4...
Baixando temporada 5...
Baixando temporada 6...
Baixando temporada 7...
Baixando temporada 8...
Baixando temporada 9...
Baixando temporada 10...

Total de episódios coletados: 235


In [12]:
df_omdb = pd.DataFrame(todos_episodios)

# Corrige os tipos de dados
df_omdb['rating_imdb'] = pd.to_numeric(df_omdb['rating_imdb'], errors='coerce')
df_omdb['votos_imdb']  = pd.to_numeric(df_omdb['votos_imdb'].str.replace(',', ''), errors='coerce')

print(df_omdb.head())
print(f'\nShape: {df_omdb.shape}')

# Salva como CSV processado
df_omdb.to_csv('../data/processed/episodios_omdb.csv', index=False)
print('Arquivo salvo em data/processed/episodios_omdb.csv')

   temporada  episodio                                          titulo  \
0          1         1            The One Where Monica Gets a Roommate   
1          1         2            The One with the Sonogram at the End   
2          1         3                          The One with the Thumb   
3          1         4              The One with George Stephanopoulos   
4          1         5  The One with the East German Laundry Detergent   

  data_estreia  rating_imdb  votos_imdb duracao  \
0  22 Sep 1994          8.1     11059.0  22 min   
1  29 Sep 1994          7.9      8351.0  22 min   
2  06 Oct 1994          8.0      8219.0  22 min   
3  13 Oct 1994          7.9      7927.0  22 min   
4  20 Oct 1994          8.3      7480.0  22 min   

                                                plot  
0  Monica and the gang introduce Rachel to the "r...  
1  Ross finds out his ex-wife is pregnant. Rachel...  
2  Monica becomes irritated when everyone likes h...  
3  Joey and Chandler take Ro